# 06 Qwen Prompting

Generated from `notebooks/ledgar_clause_classification_pipeline.ipynb`.

Source cell indices: `1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 35, 36, 37`.

- Runs Qwen prompting when RUN_QWEN_BASELINE is True and runtime requirements are met.


## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook.



Importing Libraries and Modules

In [ ]:
from pathlib import Path

import importlib.util
import os
import subprocess
import sys
import numpy as np
import pandas as pd

import json
import math
import re

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

File Setup

In [ ]:
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

In [ ]:

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )

In [ ]:
# Find the project root and add it to sys.path so that imports work, even if the notebook is opened in a subfolder or outside the project.
PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# List of (import_name, pip_name) for packages commonly used in notebooks. pip_name can be None if it's the same as import_name.
REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]

# In Colab, install all requirements from requirements-colab.txt if any are missing, to avoid multiple pip installs.
def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Check for missing imports before installing requirements in Colab, to avoid unnecessary pip installs and speed up notebook startup.
missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"


if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)

Custom Modules and Libraries

In [ ]:
from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import create_ledgar_eda, preprocess_ledgar
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.sequence_model import SequenceModelConfig, train_sequence_classifier
from modules.transformer_model import train_transformer_classifier
from modules.transformer_hpt import TransformerHPTConfig, run_two_stage_transformer_hpt
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis
from modules.wandb_reporting import finish_wandb_run, log_wandb_outputs, start_wandb_run



| Setting | Value | Purpose |
|---|---:|---|
| `SEED` | `42` | Makes sampling, baseline randomness, and train/test helper behavior reproducible. |
| `DATASET_NAME` | `LEDGAR` | Keeps the main experiment scoped to LEDGAR clause classification. |
| `TOP_K_LABELS` | `20` | Restricts the task to the 20 most frequent training labels for a manageable coursework experiment. |
| `RUN_CLASSICAL_MODELS` | `True` | Enables TF-IDF model experiments. |
| `RUN_TRANSFORMER` | `True` | Attempts transformer fine-tuning only when the runtime can support it. |
| `RUN_QWEN_BASELINE` | `True` | Attempts Qwen prompting only when GPU/model loading is available. |
| `RUN_AGENTIC_EXTENSION` | `True` | Enables a small review workflow demonstration, not an autonomous agent. |
| `RUN_WANDB` | `True` | Sends metrics and safe artifacts to W&B when credentials are available. |

Model and feature hyperparameters declared here:

| Component | Hyperparameters |
|---|---|
| TF-IDF search | `max_features` in `[10000, 30000]`; `ngram_range` in `[(1, 1), (1, 2)]`; `lowercase=True`; `stop_words=None` |
| Transformer | `distilbert-base-uncased`; `max_length=256` |
| Optional legal transformer | `nlpaueb/legal-bert-base-uncased` can be substituted manually if GPU resources allow |
| Qwen prompting | `Qwen/Qwen2.5-3B-Instruct`; test sample size `200`; one few-shot example per class when available |
| W&B logging | Uses `WANDB_API_KEY` from Colab Secrets or the environment; text-containing prediction/error tables are not uploaded unless `WANDB_LOG_TEXT_TABLES=True` |

Explainability note: keeping all configuration values in one cell makes it clear which choices affect runtime cost, model capacity, and evaluation scope.

In [ ]:
SEED = 42

DATASET_NAME = "LEDGAR"

TOP_K_LABELS = 20

MAX_FEATURES_LIST = [10000, 30000]

NGRAM_RANGES = [(1, 1), (1, 2)]

RUN_CLASSICAL_MODELS = True

RUN_TRANSFORMER = True

RUN_TRANSFORMER_HPT = False

HPT_RANDOM_TRIALS = 8

HPT_BAYES_TRIALS = 8

RUN_QWEN_BASELINE = True

RUN_AGENTIC_EXTENSION = True

RUN_NAIVE_BAYES = True

RUN_SEQUENCE_MODEL = False

RUN_WANDB = True

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ledgar-clause-classification")

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None

WANDB_MODE = os.environ.get("WANDB_MODE", "online")

WANDB_LOG_ARTIFACTS = True

WANDB_LOG_TEXT_TABLES = False

WANDB_LOG_MODEL_FILES = False

TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"

OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

MAX_TRANSFORMER_LENGTH = 256

QWEN_EVAL_SAMPLE_SIZE = 200

QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1


DOWNLOAD_LEDGAR_IF_MISSING = True

DOWNLOAD_CUAD_IF_MISSING = True

USE_HF_CACHE = True

FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)


Weights and Biases Setup

In [ ]:
print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")

In [ ]:

try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")
except Exception:
    print("PyTorch is unavailable. Transformer/Qwen sections will skip if they require it.")

wandb_run = start_wandb_run(
    enabled=RUN_WANDB,
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    group="ledgar-coursework",
    tags=["ledgar", "legal-clause-classification", "coursework"],
    config={
        "seed": SEED,
        "dataset_name": DATASET_NAME,
        "top_k_labels": TOP_K_LABELS,
        "run_classical_models": RUN_CLASSICAL_MODELS,
        "run_transformer": RUN_TRANSFORMER,
        "run_transformer_hpt": RUN_TRANSFORMER_HPT,
        "hpt_random_trials": HPT_RANDOM_TRIALS,
        "hpt_bayes_trials": HPT_BAYES_TRIALS,
        "run_qwen_baseline": RUN_QWEN_BASELINE,
        "run_agentic_extension": RUN_AGENTIC_EXTENSION,
        "run_naive_bayes": RUN_NAIVE_BAYES,
        "run_sequence_model": RUN_SEQUENCE_MODEL,
        "transformer_model_name": TRANSFORMER_MODEL_NAME,
        "max_transformer_length": MAX_TRANSFORMER_LENGTH,
        "qwen_model_name": QWEN_MODEL_NAME,
        "qwen_eval_sample_size": QWEN_EVAL_SAMPLE_SIZE,
        "device": str(DEVICE),
        "log_text_tables": WANDB_LOG_TEXT_TABLES,
        "log_model_files": WANDB_LOG_MODEL_FILES,
    },
    mode=WANDB_MODE,
)

WANDB_ACTIVE = wandb_run is not None


## 2. Dataset Download and Raw Setup

This stage obtains the raw datasets without training or preprocessing models. LEDGAR remains the main classification dataset. CUAD is treated separately because it is structured as a contract-review question-answering/span-extraction dataset rather than a direct clause-classification dataset.

Data governance choices:

- LEDGAR is loaded from Hugging Face with `load_dataset("coastalcph/lex_glue", "ledgar")` when local JSONL files are missing.
- Official LEDGAR train, validation, and test splits are preserved when available.
- Raw LEDGAR split exports are saved under `data/raw/lexglue_ledgar/` as JSONL files.
- CUAD raw files are downloaded from `theatticusproject/cuad` when available, but CUAD is not merged with LEDGAR.
- If CUAD is missing, the notebook prints a clear message and continues with LEDGAR.

Inputs and outputs:

| Input | Output |
|---|---|
| Hugging Face LEDGAR or local JSONL | `ledgar_raw_splits` dictionary with train/validation/test DataFrames |
| Optional CUAD raw files | `cuad_clause_df` containing extracted span-level examples for optional inspection |

This stage deliberately does not select labels, encode classes, train models, or compute metrics.


CAUD Dataset

In [ ]:
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

## 3. LEDGAR Preprocessing and EDA

This stage converts raw LEDGAR into a consistent clause-classification schema used by all later experiments.

Preprocessing decisions:

- Standard schema: `text`, `label`, `label_id`, `split`, `source_dataset`.
- Text cleaning is intentionally light: whitespace is normalised and leading/trailing spaces are stripped.
- Legal punctuation and stopwords are retained because they may carry meaning in contractual language.
- Empty or malformed examples are removed.
- Exact duplicate `text` plus `label` pairs are removed to reduce repeated rows.
- The top `TOP_K_LABELS=20` labels are selected using the training split only, preventing validation/test leakage in label selection.
- Label IDs are assigned after filtering so every model uses the same `label2id` and `id2label` mappings.

EDA outputs generated here:

| Output | Purpose |
|---|---|
| `class_distribution.png` | Shows imbalance across the selected labels. |
| `clause_length_histogram.png` | Shows clause length variation, useful for interpreting transformer truncation risk. |
| `dataset_split_summary.csv` | Records split sizes and class counts. |
| `examples_per_label.jsonl` | Provides qualitative examples for annotation and ambiguity inspection. |

The processed JSONL files are saved under `data/processed/` and become the controlled inputs for all model sections.


In [ ]:
processed_splits, label2id, id2label = preprocess_ledgar(

    ledgar_raw_splits,

    paths,

    top_k_labels=TOP_K_LABELS,

    dataset_name=DATASET_NAME,
)

split_summary = create_ledgar_eda(processed_splits, paths.results_dir)


if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))

else:
    train_df = validation_df = test_df = pd.DataFrame(columns=["text",
                                                               "label",
                                                               "label_id",
                                                               "split",
                                                               "source_dataset"])

    label_names = []


    print("Main LEDGAR experiment cannot run without LEDGAR data.")

## 4. Shared Result State

This short stage creates shared containers used by the later model sections.

- `completed_results` stores one row per completed or skipped model run.
- `prediction_tables` stores per-example predictions for error analysis.
- `trained_models` stores reusable fitted model objects when available.

Governance purpose: every model section appends to the same result structure, so the final comparison table is generated from actual run outputs rather than manually entered values.


Variable Initialization

In [ ]:
completed_results = []

prediction_tables = {}

trained_models = {}

## 9. Qwen2.5-Instruct Prompting Baseline

This stage optionally evaluates an instruction-tuned language model as a prompting baseline. Qwen is not fine-tuned; it is only prompted to classify clauses into the fixed LEDGAR label set.

Prompting setup:

| Mode | Description |
|---|---|
| Zero-shot | Provides the clause text and full list of allowed labels. |
| Static few-shot | Adds training examples only; validation and test examples are never used as demonstrations. |
| Retrieval few-shot | Retrieves similar examples from the training split only. |

Generation and parsing controls:

| Setting | Value |
|---|---:|
| Model | `Qwen/Qwen2.5-3B-Instruct` |
| Evaluation sample | up to `200` test examples, sampled with `SEED=42` |
| Decoding | deterministic, `do_sample=False` |
| New tokens | `max_new_tokens=20` |
| Output requirement | return exactly one allowed label |
| Parser | exact allowed-label match after whitespace/case normalisation |
| Invalid outputs | marked as `INVALID_PREDICTION` and reported separately |

Governance note: this is not directly equivalent to supervised fine-tuning. The prompted model has different pretraining and task setup, so results should be interpreted as a separate baseline rather than a perfectly fair model-family comparison.


In [ ]:


qwen_output = run_qwen_baseline(

    train_df,
    # Training data for Qwen prompting (used to create few-shot examples)

    test_df,
    # Test data for Qwen prompting (used for evaluation)

    label2id,
    # Mapping from label names to label IDs, which may be needed for formatting prompts or interpreting outputs

    id2label,
    # Mapping from label IDs to label names, which may be needed for formatting prompts or interpreting outputs

    paths.results_dir,
    # Directory to save results and artifacts related to the Qwen baseline

    model_name=QWEN_MODEL_NAME,
    # Name of the Qwen model to use for prompting (e.g., "Qwen/Qwen2.5-3B-Instruct")

    label_names=label_names,
    # List of label names corresponding to the label IDs, which may be needed for formatting prompts or interpreting outputs

    eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    # Number of test samples to evaluate on for the Qwen baseline (e.g., 200)

    few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    # Number of few-shot examples to include per class in the prompt for Qwen (e.g., 1)

    dataset_name=DATASET_NAME,
    # Name of the dataset (used for logging and artifact naming)

    seed=SEED,
    # Random seed for reproducibility, which may be used for sampling evaluation examples or shuffling data

    run_qwen=RUN_QWEN_BASELINE,
    # Whether to run the Qwen baseline (if False, the function may skip execution and return None or a skip result)
)



In [ ]:

completed_results.extend(qwen_output["results"])

qwen_predictions_df = qwen_output["predictions"]

qwen_invalid_outputs_df = qwen_output["invalid_outputs"]

qwen_model = qwen_output["model"]

qwen_tokenizer = qwen_output["tokenizer"]

if qwen_output["results"]:
    display(pd.DataFrame(qwen_output["results"])[[
        "model_name",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "notes"]])

#
# If qwen_output contains valid results,
#   append the results to completed_results,
#   store the predictions, invalid outputs, model, and tokenizer in respective variables,
#   and display a DataFrame with the Qwen baseline results showing:
#       "model_name",
#       "accuracy",
#       "macro_f1",
#       "weighted_f1",
#       "notes" columns.
#